In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
# Configuracion
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

Usando dispositivo: cpu


In [3]:
# 1. DATOS DE EJEMPLO (comentarios en español)
comentarios = [
    # Positivos (1)
    "Me encantó este producto, es excelente",
    "Muy buen servicio, lo recomiendo totalmente",
    "La atención al cliente fue increíble",
    "Excelente calidad, superó mis expectativas",
    "Me siento muy satisfecho con mi compra",
    "El producto llegó antes de lo esperado",
    "Totalmente recomendado, funciona perfecto",
    "La mejor compra que he hecho este año",
    "El personal es muy amable y profesional",
    "Volvería a comprar sin dudarlo",
    
    # Negativos (0)
    "Pésimo servicio, no lo recomiendo",
    "El producto llegó roto y en mal estado",
    "Muy mala experiencia, no volveré a comprar",
    "La atención al cliente es terrible",
    "No funciona como esperaba, decepcionado",
    "El envío tardó mucho más de lo debido",
    "Producto de baja calidad, no vale lo que cuesta",
    "Me arrepiento de haber comprado esto",
    "El servicio al cliente no responde",
    "No cumple con lo prometido, muy malo"
]


In [4]:
etiquetas = [1] * 10 + [0] * 10  # 1 para positivos, 0 para negativos
etiquetas

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [5]:
# Crea dataFrame
df = pd.DataFrame({'comentarios': comentarios, 'sentimiento': etiquetas})
df

,comentarios,sentimiento
0,"Me encantó este producto, es excelente",1
1,"Muy buen servicio, lo recomiendo totalmente",1
2,La atención al cliente fue increíble,1
3,"Excelente calidad, superó mis expectativas",1
4,Me siento muy satisfecho con mi compra,1
5,El producto llegó antes de lo esperado,1
6,"Totalmente recomendado, funciona perfecto",1
7,La mejor compra que he hecho este año,1
8,El personal es muy amable y profesional,1
9,Volvería a comprar sin dudarlo,1


## DataSet personalizado para PyTorch

In [6]:
class SentimentDataset(Dataset):
    def __init__(self, comentarios, etiquetas, tokenizer, max_len=120):
        self.comentarios = comentarios
        self.etiquetas = etiquetas
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.comentarios)

    def __getitem__(self, idx):
        texto = str(self.comentarios[idx])
        etiqueta = self.etiquetas[idx]

        # Tokenizar el texto
        encoding = self.tokenizer(
            texto,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(int(etiqueta), dtype=torch.long)
        }

In [7]:
# 3. Preparar datos
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [8]:
# Dividir datos en entrenamiento y puruebas
x_train, x_test, y_train, y_test = train_test_split(
    df['comentarios'].values,
    df['sentimiento'].values,
    test_size=0.3,
    random_state=23,
    stratify=df['sentimiento'].values
)

In [9]:
print(f'Tamaño del conjunto de entrenamiento: {len(x_train)}')
print(f'Tamaño del conjunto de prueba: {len(x_test)}')

Tamaño del conjunto de entrenamiento: 14
Tamaño del conjunto de prueba: 6


In [10]:
train_dataset = SentimentDataset(x_train, y_train, tokenizer)
test_dataset = SentimentDataset(x_test, y_test, tokenizer)
test_dataset


In [11]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=True)

## Entrenamiento del modelo

In [12]:
modelo = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
modelo = modelo.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
optimizador = AdamW(modelo.parameters(), lr=2e-5)

In [14]:
# Funcion de entrenamiento
def entrenar_modelo(modelo, train_dataloader, test_dataloader, optimizer, epocas=5):
    train_losses = []
    test_accuracies = []
    for epoca in range(epocas):
        print(f'Epoca {epoca + 1}/{epocas}: ')
        print('=' * 50)
        # Modo entrenamiento
        modelo.train()
        total_loss = 0
        
        for batch in tqdm(train_dataloader, desc='Entrenando'):
            # Mover datos al dispositivo
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            ## Pasar
            optimizer.zero_grad()
            outputs = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss
            
            loss.backward()
            optimizer.step()

        avg_loss = total_loss / len(train_dataloader)
        train_losses.append(avg_loss)
            
        # Evaluacion
        accuracy = evaluar_modelo(modelo, test_dataloader)
        test_accuracies.append(accuracy)
        print(f'Promedio de perdidas: {avg_loss:.4f} | Accuracy en prueba: {accuracy:.4f}')
            
    return train_losses, test_accuracies

In [15]:
def evaluar_modelo(modelo, test_dataloader):
    modelo.eval()
    predicciones = []
    verdaderos = []
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc='Evaluando'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = modelo(input_ids=input_ids, attention_mask=attention_mask)
            _, pred = torch.max(outputs.logits, dim=1)
            predicciones.extend(pred.cpu().numpy())
            verdaderos.extend(labels.cpu().numpy())
            
    accuracy = accuracy_score(verdaderos, predicciones)
    return accuracy
            

In [16]:
# Entranar el modelo
train_losses, test_accuracies = entrenar_modelo(
    modelo, train_dataloader, test_dataloader, optimizador, epocas=5
)

Epoca 1/5: 


Evaluando: 100%|██████████| 2/2 [00:01<00:00,  1.94it/s]


Promedio de perdidas: 0.8000 | Accuracy en prueba: 0.5000
Epoca 2/5: 


Evaluando: 100%|██████████| 2/2 [00:00<00:00,  2.08it/s]


Promedio de perdidas: 0.7484 | Accuracy en prueba: 0.1667
Epoca 3/5: 


...

Promedio de perdidas: 0.6661 | Accuracy en prueba: 0.5000
Epoca 4/5: 


Evaluando: 100%|██████████| 2/2 [00:00<00:00,  2.11it/s]


Promedio de perdidas: 0.6885 | Accuracy en prueba: 0.5000
Epoca 5/5: 


Evaluando: 100%|██████████| 2/2 [00:00<00:00,  2.10it/s]

Promedio de perdidas: 0.5657 | Accuracy en prueba: 0.5000


In [17]:
# Evaluación final detallada
modelo.eval()
all_predicciones = []
all_verdaderos = []
all_probabilidades = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = modelo(
        input_ids=input_ids,
        attention_mask=attention_mask
        )

        # Obtener probabilidades con softmax
        probabilidades = torch.nn.functional.softmax(outputs.logits, dim=1)
        _, preds = torch.max(outputs.logits, dim=1)

        all_predicciones.extend(preds.cpu().tolist())
        all_verdaderos.extend(labels.tolist())
        all_probabilidades.extend(probabilidades.cpu().tolist())

        # Reporte de clasificación
        print("\n" + "="*50)
        print("REPORTE DE CLASIFICACIÓN")
        print("="*50)
        print(classification_report(
        all_verdaderos,
        all_predicciones,
        target_names=['Negativo', 'Positivo']
        ))

# Mostrar algunos ejemplos con predicciones
print("\n" + "="*50)
print("EJEMPLOS DE PREDICCIONES")
print("="*50)
indices_prueba = np.random.choice(len(x_test), size=5, replace=False)
for idx in indices_prueba:
    comentario = x_test[idx]
    real = "POSITIVO" if y_test[idx] == 1 else "NEGATIVO"
    pred = "POSITIVO" if all_predicciones[idx] == 1 else "NEGATIVO"
    prob_pos = all_probabilidades[idx][1]
    prob_neg = all_probabilidades[idx][0]

    print(f"\nComentario: {comentario}")
    print(f"Real: {real}")
    print(f"Predicción: {pred}")
    print(f"Probabilidad Positivo: {prob_pos:.4f}")
    print(f"Probabilidad Negativo: {prob_neg:.4f}")
    print(f"{'✓' if real == pred else '✗'} Correcto: {real == pred}")


REPORTE DE CLASIFICACIÓN
              precision    recall  f1-score   support

    Negativo       0.33      0.50      0.40         2
    Positivo       0.00      0.00      0.00         2

    accuracy                           0.25         4
   macro avg       0.17      0.25      0.20         4
weighted avg       0.17      0.25      0.20         4


REPORTE DE CLASIFICACIÓN
              precision    recall  f1-score   support

    Negativo       0.50      0.67      0.57         3
    Positivo       0.50      0.33      0.40         3

    accuracy                           0.50         6
   macro avg       0.50      0.50      0.49         6
weighted avg       0.50      0.50      0.49         6


EJEMPLOS DE PREDICCIONES

Comentario: Excelente calidad, superó mis expectativas
Real: POSITIVO
Predicción: POSITIVO
Probabilidad Positivo: 0.6089
Probabilidad Negativo: 0.3911
✓ Correcto: True

Comentario: Totalmente recomendado, funciona perfecto
Real: POSITIVO
Predicción: POSITIVO
Probabil

## Función para nuevos comentarios

In [18]:
def predecir_sentimiento(texto, model, tokenizer, device):
    """
    Predice el sentimiento de un nuevo comentario
    """
    modelo.eval()
    
    # Tokenizar
    encoding = tokenizer(
        texto,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # Mover a dispositivo
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predecir
    with torch.no_grad():
        outputs = modelo(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        probabilidades = torch.nn.functional.softmax(outputs.logits, dim=1)
        prediccion = torch.argmax(probabilidades, dim=1)
        
        prob_pos = probabilidades[0][1].item()
        prob_neg = probabilidades[0][0].item()
    
    sentimiento = "POSITIVO" if prediccion.item() == 1 else "NEGATIVO"
    confianza = max(prob_pos, prob_neg)
    
    return {
        'sentimiento': sentimiento,
        'confianza': confianza,
        'probabilidad_positivo': prob_pos,
        'probabilidad_negativo': prob_neg
    }

## Listado de nuevos comentarios

In [19]:
# Probar con nuevos comentarios
print("\n" + "="*50)
print("PREDICCIONES EN TIEMPO REAL")
print("="*50)
nuevos_comentarios = [
    "Este producto es increíble, me encantó",
    "Una verdadera porquería, no funciona",
    "Más o menos, podría ser mejor",
    "Excelente atención, muy recomendable",
    "No me gustó para nada, muy decepcionante"
]

for comentario in nuevos_comentarios:
    resultado = predecir_sentimiento(comentario, modelo, tokenizer, device)
    print(f"\nComentario: {comentario}")
    print(f"Sentimiento: {resultado['sentimiento']}")
    print(f"Confianza: {resultado['confianza']:.2%}")
    print(f"Positivo: {resultado['probabilidad_positivo']:.2%}, "
          f"Negativo: {resultado['probabilidad_negativo']:.2%}")


PREDICCIONES EN TIEMPO REAL

Comentario: Este producto es increíble, me encantó
Sentimiento: POSITIVO
Confianza: 64.85%
Positivo: 64.85%, Negativo: 35.15%

Comentario: Una verdadera porquería, no funciona
Sentimiento: POSITIVO
Confianza: 54.25%
Positivo: 54.25%, Negativo: 45.75%

Comentario: Más o menos, podría ser mejor
Sentimiento: POSITIVO
Confianza: 57.96%
Positivo: 57.96%, Negativo: 42.04%

Comentario: Excelente atención, muy recomendable
Sentimiento: POSITIVO
Confianza: 61.45%
Positivo: 61.45%, Negativo: 38.55%

Comentario: No me gustó para nada, muy decepcionante
Sentimiento: POSITIVO
Confianza: 63.99%
Positivo: 63.99%, Negativo: 36.01%


In [23]:
comentario = 'Me encanta, lo volvería a comprar'
resultado = predecir_sentimiento(comentario, modelo, tokenizer, device)
print(f"\nComentario: {comentario}")
print(f"Sentimiento: {resultado['sentimiento']}")
print(f"Confianza: {resultado['confianza']:.2%}")
print(f"Positivo: {resultado['probabilidad_positivo']:.2%}, "
      f"Negativo: {resultado['probabilidad_negativo']:.2%}")


Comentario: Me encanta, lo volvería a comprar
Sentimiento: POSITIVO
Confianza: 63.36%
Positivo: 63.36%, Negativo: 36.64%
